# 上下文和记忆

## 上下文工程

在 LangChain 中， 记忆(Memory) 就是专门负责“ 存储历史交互信息 ”的组件，核心作用是「 保存上下文 」和「 提供上下文 」，让LLM在每次响应时，都能“看到”之前的对话内容。

上下文工程(Context Engineering) 负责 “合理组织”这些记忆和任务信息 ，让LLM的响应更连贯、更贴合需求。这也是Agent能实现 复杂多轮交互 的核心基础。

![image.png](images/context.png)

## 记忆的分类

- 短期记忆 ：会话级记忆，作用范围是单个对话线程内，一旦开始新对话(更换线程id)，记忆即消失
- 长期记忆：跨会话级记忆，在会话间存储用户特定或应用级数据， 并在 会话线程间共享 。它可以随时在任何线程中被调用。记忆的范围是任意自定义命名空间，而不仅仅是单一线程 ID。

## 记忆的管理

在LangChain v1.x版本中，Agent是构建在LangGraph图结构之上的，通过上文提到的state和store构建记忆系统。使用更简单、功能更统一。
- state：短期记忆对象，以 会话 为单位组织，包含当前会话的所有消息记录以及自定义信息。
- store：长期记忆对象， 跨会话持久化 的数据，通常需要结合向量数据库或外部存储实现。

# 短期记忆

LangChain1.x 的短期记忆是三者的组合：
State（会话内部状态） + Checkpointer（持久化机制） + Thread ID（会话作用域）

- State ：默认 存储历史消息列表messages ，通过State 管理历史消息
- Checkpointer ：负责将State 作为检查点持久化保存，检查点是某个时刻的State 快照
- Thread ID ：用于唯一标识State ，LangChain运行时会按照 thread_id 读写State快照

### 基于内存的持久化器

In [1]:
# 没有记忆
from langchain.agents import  create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

agent=create_agent(model=model,
                   tools=[])

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]
})
print(f"Agent: {response1['messages'][-1].content}")
print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]
})
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的吗？

第二轮对话：
Agent: 我不知道你叫什么名字，因为你还没有告诉我。你可以告诉我你想让我怎么称呼你吗？


In [8]:
# 拥有记忆

from langgraph.checkpoint.memory import InMemorySaver

checkpointer=InMemorySaver()

agent=create_agent(model=model,
                   tools=[],
                   checkpointer=checkpointer #添加内存管理
                     )

config={
    "configurable":{
        "thread_id":"1"
    }
}

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config=config
)
print(f"Agent: {response1['messages'][-1].content}")
print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config
)
print(f"Agent: {response2['messages'][-1].content}")



第一轮对话：
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的吗？

第二轮对话：
Agent: 你叫张三。


### 只需传入 checkpointer 和 config，Agent 就能自然具备连续对话能力。

In [3]:
from rich import print as rprint

latest_state = agent.get_state(config)
rprint(latest_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='4108e897-cfa1-45c4-b22d-9680748397ea'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。有什么我可以帮你的吗？',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 
'我们需要回答中文。用户只是说“我叫张三”。应该介绍/回应。可以礼貌说你好张三，有什么可以帮你。无需复杂。注意不要泄露
。最终简洁。'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 51,
                        'prompt_tokens': 32,
                        'total_tokens': 83,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 36,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 32
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': 'cc60da07-f95c-412c-b8dc-279eeb19f06c',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a1e979-daa7-705c-9ed5-6b7cd7b26d91-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 32,
                    'output_tokens': 51,
                    'total_tokens': 83,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 36}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='bf86085c-02f5-4cf5-b788-62f2c7285d02'
            ),
            AIMessage(
                content='你叫张三。',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 
'我们需要回答用户中文问“我叫什么？”根据对话历史，用户之前说“我叫张三”。所以回答：你叫张三。简洁。需要确认。'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 37,
                        'prompt_tokens': 53,
                        'total_tokens': 90,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 32,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 53
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': '0d3236f6-47f0-4776-8ddc-c714224a6234',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a1e979-df8d-729f-8c96-da750071b184-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 53,
                    'output_tokens': 37,
                    'total_tokens': 90,
                    'input_token_details': {'cache_read': 0},
                 

In [9]:
print("\n第三轮对话：")
response3 = agent.invoke({
    "messages": [HumanMessage("我刚才问了什么问题？")]},
    config=config
)
print(response3["messages"][-1].content)


第三轮对话：
你刚才问的是：“我叫什么？”


In [13]:
# 更新线程id，则会重新开始对话

config={
    "configurable":{
        "thread_id":"3"
    }
}

response4=agent.invoke({
    "messages":[HumanMessage("我记得我叫什么吗")]
    },
    config=config
)

print(response4["messages"][-1].content)


我不记得你叫什么。我这边没有你的个人信息，也没有更早的聊天记录；如果你愿意告诉我，我可以在这段对话里这样称呼你。


# 核心步骤

## 1.初始化记忆引擎
    checkpointer=InMemorySaver()创建一个内存级的记忆存储。
    InMemorySaver内存中保存，进程结束就丢失数据，适合测试。生产环境可换成数据库持久化的 SqliteSaver 、 PostgresSaver 等
## 2.绑定 Agent
    在 create_agent 时传入 checkpointer ，让 Agent 具备状态存储能力。
## 3. 设定会话 ID
    通过 config = {"configurable": {"thread_id": "1"}} 为每次调用指定线程标识。同一个 thread_id 共享记忆，不同 thread_id 完全隔离。

thread_id 是记忆管理的核心开关：在会话2里询问会话1的会话信息，Agent 会表示不知道——因为双方记忆空间完全隔离。

### 工作原理

自动追加历史
```
# 你只需要传新消息
agent.invoke(
    {"messages": [{"role": "user", "content": "新问题"}]},
    config
)
```

此时，checkpointer自动： 读取之前的历史，追加新消息，调用模型(传入完整历史)，保存新历史

- 当程序重启时，内存中的历史会丢失

解决方法 ：持久化  SQLite,postgreSQL

- 内存也会为无限增长

默认情况下，InMemorySaver 会保存所有消息。
    问题：
   1. 消息越来越多（无限增长，需要管理上下文
   2. token消耗增加，甚至会超过模型的 token 限制
   3. 响应速度变慢、成本增加

解决方法：上下文管理

- 清空某个会话的历史

临时方案：
   1. 使用新的thread_id
   2. 重新创建agent


## 基于外部存储介质的持久化器

如果将 状态检查点(checkpointer) 保存在内存， 进程结束 则状态丢失，生产环境不可接受。因此，生产环境要用持久化的外部存储介质，如PostgreSQL。

In [6]:
from langgraph.checkpoint.postgres import PostgresSaver
from langchain.agents import  create_agent
from langchain.messages import  HumanMessage
from langchain.chat_models import  init_chat_model
import os
from dotenv import  load_dotenv

load_dotenv(override=True)

model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

DB_URL = os.getenv("DATABASE_URL")  # 连接串放 .env，别硬编码

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # 初始化PostgreSQL数据库
    checkpointer.setup()

    agent = create_agent(
        model=model,
        checkpointer=checkpointer
    )

    config={
        "configurable":{
            "thread_id":"1"
        }
    }

    response1 = agent.invoke(
        {"messages": [HumanMessage("你好，我是老王")]},
        config=config
    )
    print("=" * 30, "-> 第一次调用 <-", "=" * 30)
    for msg in response1["messages"]:
        msg.pretty_print()
    response2 = agent.invoke(
        {"messages": [HumanMessage("你好，我是谁？")]},
        config=config
    )
    print("=" * 30, "-> 第二次调用 <-", "=" * 30)
    for msg in response2["messages"]:
        msg.pretty_print()

============================== -> 第一次调用 <- ==============================
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

老王你好！很高兴认识你。有什么我可以帮你的吗？
============================== -> 第二次调用 <- ==============================
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

老王你好！很高兴认识你。有什么我可以帮你的吗？
================================ Human Message =================================

你好，我是谁？
================================== Ai Message ==================================

你是老王呀——这是你刚才告诉我的。如果还有别的称呼或身份，也可以再告诉我。


In [8]:
# 基于内存存储

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}
print("=" * 30, "-> 第一次调用 <-", "=" * 30)
response1 = agent.invoke(
    {"messages": [HumanMessage("你好，我是谁？")]},
    config=config
)
for msg in response1["messages"]:
    msg.pretty_print()
print("=" * 30, "-> 第二次调用 <-", "=" * 30)
response2 = agent.invoke(
    {"messages": [HumanMessage("我是老王")]},
    config=config
)
for msg in response2["messages"]:
    msg.pretty_print()
print("=" * 30, "-> 第三次调用 <-", "=" * 30)
response3 = agent.invoke(
    {"messages": [HumanMessage("你好，我是谁？")]},
    config=config
)

for msg in response3["messages"]:
    msg.pretty_print()

============================== -> 第一次调用 <- ==============================
================================ Human Message =================================

你好，我是谁？
================================== Ai Message ==================================

你好！我无法知道你的真实身份，因为我们现在是匿名对话。你是正在和我聊天的朋友；如果你愿意，可以告诉我你想让我怎么称呼你。
============================== -> 第二次调用 <- ==============================
================================ Human Message =================================

你好，我是谁？
================================== Ai Message ==================================

你好！我无法知道你的真实身份，因为我们现在是匿名对话。你是正在和我聊天的朋友；如果你愿意，可以告诉我你想让我怎么称呼你。
================================ Human Message =================================

我是老王
================================== Ai Message ==================================

你好，老王！很高兴认识你。接下来你想聊点什么，或者需要我帮你做什么？
============================== -> 第三次调用 <- ==============================
================================ Human Message =================================

你好，我是谁？
==================

In [10]:
#基于外部存储器存储

from langgraph.checkpoint.postgres import PostgresSaver
from langchain.agents import  create_agent
from langchain.messages import  HumanMessage
from langchain.chat_models import  init_chat_model
import os
from dotenv import  load_dotenv

load_dotenv(override=True)

model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

DB_URL = os.getenv("DATABASE_URL")  # 连接串放 .env，别硬编码

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # 初始化PostgreSQL数据库
    checkpointer.setup()

    agent = create_agent(
        model=model,
        checkpointer=checkpointer
    )

    config={
        "configurable":{
            "thread_id":"1"
        }
    }

    print("=" * 30, "-> 第一次调用 <-", "=" * 30)
    response1 = agent.invoke(
        {"messages": [HumanMessage("你好，我是谁啊？")]},
        config
    )
    for msg in response1["messages"]:
        msg.pretty_print()
    print("=" * 30, "-> 第二次调用 <-", "=" * 30)
    response2 = agent.invoke(
        {"messages": [HumanMessage("我是老王~")]},
        config
    )
    for msg in response2["messages"]:
        msg.pretty_print()
    print("=" * 30, "-> 第三次调用 <-", "=" * 30)
    response3 = agent.invoke(
        {"messages": [HumanMessage("你好，我是谁？？")]},
        config
    )

    for msg in response3["messages"]:
        msg.pretty_print()




============================== -> 第一次调用 <- ==============================
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

老王你好！很高兴认识你。有什么我可以帮你的吗？
================================ Human Message =================================

你好，我是谁？
================================== Ai Message ==================================

你是老王呀——这是你刚才告诉我的。如果还有别的称呼或身份，也可以再告诉我。
================================ Human Message =================================

你好，我是谁啊？
================================== Ai Message ==================================

你好！根据你刚才说的，你是老王。要我再帮你确认别的身份信息吗？
================================ Human Message =================================

我是老王~
================================== Ai Message ==================================

哈哈，好嘞老王~ 记住了！有什么想聊的或需要帮忙的，随时说。
================================ Human Message =================================

你好，我是谁？？
=======================

即便重新创建 Saver() ，只要 thread_id 一致，历史状态就可以和当前调用串联起来。

s1. InMemorySaver()将状态持久化到内存， 进程结束或重建Saver() 则历史状态丢失
2. 基于外部存储介质（如PostgreSQL）的持久化器，其存储的状态不会随进程终止而丢失，只要 不
显式删除历史状态 ，即可通过 thread_id 加载历史状态。

# 上下文管理

随着对话的进行，历史消息不断累积， state会持续增长 ，为模型带来挑战：

1. LLM的 上下文窗口是有限的 ，完整历史可能无法装入LLM的上下文窗口，导致上下文丢失或错误。
2. 即便模型的上下文窗口够大，多数LLM在长上下文场景仍然表现不佳。模型会 被陈旧或离题的内容“分散注意力” 。
3. 同时，会带来 高昂的token花费 。

此时需要对上下文进行管理：对历史记录进行压缩、清理、重组等

## 消息裁剪

调用模型前裁剪上下文。

In [11]:
from langchain.chat_models import  init_chat_model
from dotenv import  load_dotenv
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

load_dotenv(override=True)

model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

@before_model
def trim_messages(state:AgentState,runtime:Runtime)->dict[str,Any] | None:
    messages=state["messages"]
    if len(messages)<=3:
        return None
    first_message=messages[0]
    recent_messages=messages[-3:] if len(messages)%2==0 else messages[-4:]
    new_messages=[first_message]+recent_messages

    return {
        "messages":[
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)
for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

好的，老王！从现在起我就是小王。有什么需要我做的，尽管说。
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，老王，今天天气确实不错，适合出去走走或者办点事。您有什么安排吗？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

老王，在咱们这段对话里：**你是老王，我是小王**。

但说实在的：我是**AI 助手**，不是真实的人；你是正在和我聊天的用户。你的真实身份、姓名等信息我无法验证，也不会去猜。刚才“老王/小王”只是按你的话设定的对话角色。


## 消息删除

消息裁剪强调“在 模型调用前裁剪 消息列表，控制模型可以看到的上下文范围”，而消息删除强调 模型调用完成后将某些消息从消息列表中移除 ，永久更改状态。

In [12]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig

@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None
agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)
final_response = agent.invoke({"messages": "告诉我，你是谁？我是谁？"}, config)
for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好的，老王！从现在起我叫小王。有什么吩咐？
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，老王，今天天气确实不错，适合出去走走。您有什么安排，或者需要小王帮您做点什么？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

老王，我是你的 AI 助手；按你刚才的要求，我在对话里叫“小王”。我不是真人，也没有真实姓名，“小王”只是你给我起的称呼。

你是正在和我对话的用户。你让我称呼你“老王”，所以在这个对话里我把你当作老王；但我无法知道你在现实中的真实身份。


## 摘要

- 把早期历史压缩成摘要，再替换原始消息。
- 消息裁剪和删除都会导致上下文缺失，影响回答质量和用户体验。和它们相比，摘要是更适合长会话的折中方案：保语义，不保原文。官方推荐内置 SummarizationMiddleware 。

In [14]:
from langchain.chat_models import  init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

load_dotenv(override=True)

model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

agent=create_agent(model=model,
                   checkpointer=InMemorySaver(),
                   tools=[],
                   middleware=[SummarizationMiddleware(
                       model=model,
                       trigger=[
                           ("tokens",100),
                       ],
                       keep=("messages", 2),
                       summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
                   )])

config = {"configurable": {"thread_id": "1"}}
print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]
for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
for msg in response["messages"]:
    msg.pretty_print()
print("*" * 50)


进行多轮对话...
================================ Human Message =================================

Here is a summary of the conversation to date:

摘要：用户自称张三，是一名工程师。其余内容为大量重复的“非常长非常长的废话”，没有新增有效信息。
================================== Ai Message ==================================

你好，张三工程师。我已看到这段重复文本，关键信息是：**你叫张三，是工程师**；其余内容主要是重复的“废话”。

请问你希望我帮你做什么？例如：
- 提取/总结关键信息
- 清理重复文本
- 统计重复次数
- 改写这段文字
- 或基于它回答某个问题
================================ Human Message =================================

请总结一下我的信息
================================== Ai Message ==================================

根据目前对话，你的信息可总结为：

- **姓名**：张三
- **职业/身份**：工程师
- **其他内容**：对话中存在大量重复的“非常长非常长的废话”，没有提供新的有效个人信息。

目前可确认的有效信息就是以上这些。
**************************************************


# 长期记忆

- 短期记忆记录的是 会话级别（线程，Thread） 的数据， 会话间不共享 。而长期记忆记录的是用户特定或应用级别的数据，任何会话都可以随时访问

![image.png](images/memory.png)

长期记忆的存储是 store -> namespace -> key -> value 的四层架构。
- 第1层：Store（记忆仓库）
Store是 langgraph.store.base.BaseStore 的子类实例，由全类名可知，store是由LangGraph提供的。
常用实现类：
1. InMemoryStore ：将长期记忆存储在内存，适合测试
2. PostgresStore ：将长期记忆存储在外部的PostgreSQL数据库，适合生产环境

开发期可用 InMemoryStore；生产建议数据库后端，如 PostgresStore
- 第2层：Namespace（命名空间）
数据类型是由任意长度的 tuple[str, ...] 表示的 层级路径 。作用上很像“文件路径 / 文件夹层级”，用于
给长期记忆分组和隔离。数据类型为 字符串元组
- 第3层：Key（键）
是该 namespace 下的唯一标识，单条记忆的唯一键，数据类型为 字符串(str)

- 第4层：Value（值）
是存储的值，数据类型为 字典(dict[str, Any])

LangChain 1.2.x 的长期记忆基于 store 持久化数据，相关的API有：
1. put() ：负责写入
2. get() ：负责读取
3. search() ：负责检索


In [ ]:
## put
store.put(
    ("users", "alice", "memories"), # namespace
    "pref_food", # key
    {"category": "food", "text": "Alice likes sushi"} # value
)

In [ ]:
## get
item = my_store.get(("users", "alice", "memories"), "pref_food")
if item is not None:
     print(item.value)
# {'category': 'food', 'text': 'Alice likes sushi'}

In [15]:
# 基于InMemoryStore

from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
namespace = ("users",)
user_id = 'user-1'
username = "小蓝"
store.put(namespace, user_id, {"name": username})
print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '小蓝'}, created_at='2026-11-18T12:08:12.897875+00:00', updated_at='2026-11-18T12:08:12.897879+00:00')


In [16]:
store.put(namespace, user_id, {"name":"rr"})
print(store.get(namespace,user_id))

Item(namespace=['users'], key='user-1', value={'name': 'rr'}, created_at='2026-11-18T12:09:23.339287+00:00', updated_at='2026-11-18T12:09:23.339289+00:00')


PostgresStore不同于InMemoryStore，更改数据的逻辑是update而非覆盖，因此 created_at 固定为Item 创建时间，而 updated_at 为更新时间，二者可能不同，符合直觉。

In [17]:
import os
from langgraph.store.postgres import PostgresStore
namespace = ("users", )
user_id = "user-11"
username = "小蓝"
DB_URL = os.getenv("DATABASE_URL")  # 连接串放 .env，别硬编码
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    store.put(namespace, user_id, {"name": username})
    print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-11', value={'name': '小蓝'}, created_at='2026-11-18T20:11:03.221980+08:00', updated_at='2026-11-18T20:11:03.221980+08:00')


In [18]:
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    store.put(namespace,user_id,{"name":"rr"})
    print(store.get(namespace,user_id))

Item(namespace=['users'], key='user-11', value={'name': 'rr'}, created_at='2026-11-18T20:11:03.221980+08:00', updated_at='2026-11-18T20:12:15.467949+08:00')


### search()

```
def search(
    self,
    namespace_prefix: tuple[str, ...], #命名空间前缀，在该前缀下搜索。
    /,
    *,
    query: str | None = None,#语义检索时用于查询的自然语言。
    filter: dict[str, Any] | None = None,#过滤条件，value中的键值对组合，见下文举例。
    limit: int = 10,#可以返回item的最大条数，效果等同于SQL中的limit。
    offset: int = 0,#返回结果之前跳过的item数量。
    refresh_ttl: bool | None = None,
) -> list[SearchItem]:
```

它支持两种检索方式（对应上面的参数2、3）：
- 按 filter 做结构化过滤 ，即用 value 中的 键值 筛选符合条件的记录。
- 按 query 做语义相似度检索 ，需要将输入转换为向量。

In [19]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}
namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}
namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}
store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [20]:
print('=' * 30, '-> (users, ) <-', "=" * 30)
for item in store.search(("users", )):
    print(item)

============================== -> (users, ) <- ==============================
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-11-18T12:25:46.082173+00:00', updated_at='2026-11-18T12:25:46.082174+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-11-18T12:25:46.082220+00:00', updated_at='2026-11-18T12:25:46.082221+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-11-18T12:25:46.082266+00:00', updated_at='2026-11-18T12:25:46.082267+00:00', score=None)


In [21]:
print('=' * 30, '-> (users, Alice) <-', "=" * 30)
for item in store.search(("users", "Alice")):
    print(item)

============================== -> (users, Alice) <- ==============================
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-11-18T12:25:46.082173+00:00', updated_at='2026-11-18T12:25:46.082174+00:00', score=None)


In [22]:
print("=" * 30, '-> (users, ), filter=food <-", "=" * 30)')
for item in store.search(("users", ), filter={"food": "紫光园奶皮子酸奶"}):
    print(item)

============================== -> (users, ), filter=food <-", "=" * 30)
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-11-18T12:25:46.082173+00:00', updated_at='2026-11-18T12:25:46.082174+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-11-18T12:25:46.082266+00:00', updated_at='2026-11-18T12:25:46.082267+00:00', score=None)


In [23]:
print("=" * 30, '-> (users, ), filter=sports <-", "=" * 30)')
for item in store.search(("users", ), filter={"sports": "跑步"}):
    print(item)

============================== -> (users, ), filter=sports <-", "=" * 30)
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-11-18T12:25:46.082173+00:00', updated_at='2026-11-18T12:25:46.082174+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-11-18T12:25:46.082220+00:00', updated_at='2026-11-18T12:25:46.082221+00:00', score=None)


In [24]:
print("=" * 30, '-> (users, ), filter=course <-", "=" * 30)')
for item in store.search(("users", ), filter={"course": "数字电路与模拟电路"}):
    print(item)

============================== -> (users, ), filter=course <-", "=" * 30)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-11-18T12:25:46.082220+00:00', updated_at='2026-11-18T12:25:46.082221+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-11-18T12:25:46.082266+00:00', updated_at='2026-11-18T12:25:46.082267+00:00', score=None)


## 在工具中访问长期记忆



In [28]:
from langchain.chat_models import  init_chat_model
from langchain_core.messages import HumanMessage
from typing import NotRequired
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv

load_dotenv(override=True)

model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

store = InMemoryStore()
class CustomState(AgentState):
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """将用户信息保存在长期记忆中

    Args:
        name: 用户名

    Returns:
        str: 保存状态
    """
    runtime.store.put(("users",), runtime.state["user_id"], {"name": name})
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """

    从长期记忆中读取用户信息

    Returns:
        str: 用户信息
    """
    item = runtime.store.get(("users",), runtime.state["user_id"])
    return str(item.value) if item else "unknown"

agent = create_agent(
    model=model,
    tools=[save_user_info, get_user_info],
    store=store,
    system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
    state_schema=CustomState,
)
print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()
print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花，很高兴认识你！我先记一下你的名字。
Tool Calls:
  save_user_info (call_00_CM9EMr2e9X9PLNe2RAuv4577)
 Call ID: call_00_CM9EMr2e9X9PLNe2RAuv4577
  Args:
    name: 小花
  get_user_info (call_01_hrl2BBHRrtbWpeBX0rl22020)
 Call ID: call_01_hrl2BBHRrtbWpeBX0rl22020
  Args:
================================= Tool Message =================================
Name: save_user_info

saved
================================= Tool Message =================================
Name: get_user_info

unknown
================================== Ai Message ==================================

已经记住啦，小花～这是我们的第一次交流，之后我就能认得你了。有什么我可以帮你的吗？
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =====================

In [29]:
# 基于postgresql

from langchain.chat_models import  init_chat_model
from langchain_core.messages import HumanMessage
from typing import NotRequired
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
import os
from dotenv import load_dotenv

load_dotenv(override=True)

model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

DB_URL = os.getenv("DATABASE_URL")  # 连接串放 .env，别硬编码
class CustomState(AgentState):
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """将用户信息保存在长期记忆中

    Args:
        name: 用户名

    Returns:
        str: 保存状态
    """
    runtime.store.put(("users",), runtime.state["user_id"], {"name": name})
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """

    从长期记忆中读取用户信息

    Returns:
        str: 用户信息
    """
    item = runtime.store.get(("users",), runtime.state["user_id"])
    return str(item.value) if item else "unknown"

with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    agent = create_agent(
        model=model,
        tools=[save_user_info, get_user_info],
        store=store,
        system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
        state_schema=CustomState,
    )
    print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
    response1 = agent.invoke({
        "messages": "你好，很高兴认识你，我是小花",
        "user_id": "user-1"
    })
    for msg in response1["messages"]:
        msg.pretty_print()
    print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
    response2 = agent.invoke({
        "messages": "我是谁",
        "user_id": "user-1"
    })
    for msg in response2["messages"]:
        msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花！很高兴认识你。有什么我可以帮你的吗？
Tool Calls:
  save_user_info (call_00_ET_wll0dTFYW7PGIjZIid3w0896)
 Call ID: call_00_ET_wll0dTFYW7PGIjZIid3w0896
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

我已经记住你的名字啦，小花 😊 以后我们聊天我就能记起你。有什么想聊的或需要帮忙的，尽管告诉我！
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_NCh2ecZdP1RDY6Qgwbdc3836)
 Call ID: call_00_NCh2ecZdP1RDY6Qgwb